In [ ]:
import pandas as pd

In [ ]:
data_set = pd.read_csv('daily rate prediction.csv')
df = pd.DataFrame(data_set)
df.head()

,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,reserved_room_type,deposit_type,adr
0,6,2015,July,27,1,0,2,1,0.0,0,HB,PRT,Offline TA/TO,TA/TO,0,A,No Deposit,0.0
1,88,2015,July,27,1,0,4,2,0.0,0,BB,PRT,Online TA,TA/TO,0,A,No Deposit,76.5
2,65,2015,July,27,1,0,4,1,0.0,0,BB,PRT,Online TA,TA/TO,0,A,No Deposit,68.0
3,92,2015,July,27,1,2,4,2,0.0,0,BB,PRT,Online TA,TA/TO,0,A,No Deposit,76.5
4,100,2015,July,27,2,0,2,2,0.0,0,BB,PRT,Online TA,TA/TO,0,A,No Deposit,76.5


In [ ]:
df.isna().sum()

,0
lead_time,0
arrival_date_year,0
arrival_date_month,0
arrival_date_week_number,0
arrival_date_day_of_month,0
stays_in_weekend_nights,0
stays_in_week_nights,0
adults,0
children,4
babies,0


In [ ]:
df = df.drop(['company','agent','reservation_status','reservation_status_date','customer_type','days_in_waiting_list','deposit_type','booking_changes','previous_bookings_not_canceled','previous_cancellations','is_repeated_guest','distribution_channel','country','babies','children','adults','arrival_date_year','lead_time','is_canceled'], axis=1)

In [ ]:
# Deleting rows with 'Resort Hotel'
df = df[df['hotel'] != 'Resort Hotel']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93452 entries, 0 to 93451
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   index                        93452 non-null  int64  
 1   hotel                        93452 non-null  object 
 2   arrival_date_month           93452 non-null  object 
 3   arrival_date_week_number     93452 non-null  int64  
 4   arrival_date_day_of_month    93452 non-null  int64  
 5   stays_in_weekend_nights      93452 non-null  int64  
 6   stays_in_week_nights         93452 non-null  int64  
 7   meal                         93452 non-null  object 
 8   market_segment               93452 non-null  object 
 9   reserved_room_type           93452 non-null  object 
 10  assigned_room_type           93452 non-null  object 
 11  adr                          93452 non-null  float64
 12  required_car_parking_spaces  93451 non-null  float64
 13  total_of_special

In [ ]:
print(df['hotel'])

40060     City Hotel
40061     City Hotel
40062     City Hotel
40063     City Hotel
40064     City Hotel
             ...    
119385    City Hotel
119386    City Hotel
119387    City Hotel
119388    City Hotel
119389    City Hotel
Name: hotel, Length: 79330, dtype: object


In [ ]:
df.isna().sum()

,0
index,0
hotel,0
arrival_date_month,0
arrival_date_week_number,0
arrival_date_day_of_month,0
stays_in_weekend_nights,0
stays_in_week_nights,0
meal,0
market_segment,0
reserved_room_type,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93452 entries, 0 to 93451
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   index                        93452 non-null  int64  
 1   hotel                        93452 non-null  object 
 2   arrival_date_month           93452 non-null  object 
 3   arrival_date_week_number     93452 non-null  int64  
 4   arrival_date_day_of_month    93452 non-null  int64  
 5   stays_in_weekend_nights      93452 non-null  int64  
 6   stays_in_week_nights         93452 non-null  int64  
 7   meal                         93452 non-null  object 
 8   market_segment               93452 non-null  object 
 9   reserved_room_type           93452 non-null  object 
 10  assigned_room_type           93452 non-null  object 
 11  adr                          93452 non-null  float64
 12  required_car_parking_spaces  93451 non-null  float64
 13  total_of_special

In [ ]:
x = df.drop(['adr','lead_time','arrival_date_week_number'], axis=1)
y = df['adr']

In [ ]:
# Encoding
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

scalar = StandardScaler()
encoder = LabelEncoder()
categorical_columns = x.select_dtypes(include=['object']).columns
for column in categorical_columns:
    x[column] = encoder.fit_transform(x[column])


# Scaling
for column in x.columns:
    x[column] = scalar.fit_transform(x[[column]])


In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor

# model = DecisionTreeRegressor(random_state=42)
model = BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),n_estimators=100, random_state=42)
model.fit(x_train, y_train)

BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                 n_estimators=100, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_pred = model.predict(x_test)
print(y_pred)
mse = mean_squared_error(y_test, y_pred)
msa = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(mse)
print(msa)
print(r2)

[ 69.544       58.42692496  80.65       ... 111.30823333 110.2555
  75.1009    ]
192.38750691897678
5.9838792695725775
0.795361020722504


In [ ]:
# Predict with XGBOOST
import xgboost as xgb

model_xgd = xgb.XGBRegressor(Objective='reg:linear', random_state=42)
model_xgd.fit(x_train, y_train)

/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [17:37:51] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "Objective" } are not used.

  warnings.warn(smsg, UserWarning)


XGBRegressor(Objective='reg:linear', base_score=None, booster=None,
             callbacks=None, colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_pred = model_xgd.predict(x_test)
print(y_pred)
mse = mean_squared_error(y_test, y_pred)
msa = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(mse)
print(msa)
print(r2)

[ 70.69758   59.538208  84.8511   ...  79.92849  113.31102   80.15268 ]
176.48607108086077
7.498072707942106
0.8122750794941533
